# Project 2 – Task 1.1: Python–SQL Server Connection (AdventureWorks)

This notebook connects Python to a local SQL Server instance (`MYDELL23\SQLEXPRESS01`) using SQLAlchemy with the pyodbc driver and Windows Authentication, and verifies the connection by loading sample data from the AdventureWorks database into pandas.

## Connection checklist

| Item                      | Value / Description                               |
|---------------------------|---------------------------------------------------|
| Python version            | 3.11.3 (virtual environment `venv`)              |
| Key packages              | `sqlalchemy`, `pyodbc`, `pandas`, `ipykernel`    |
| SQL Server instance       | `MYDELL23\SQLEXPRESS01`                          |
| Database                  | `AdventureWorks` (sample OLTP database)      |
| Authentication            | Windows Authentication (`trusted_connection=yes`) |
| Test query                | `SELECT TOP 10` from `Sales.SalesOrderHeader`    |



In [1]:
import pandas as pd
from sqlalchemy import create_engine, text


In [2]:
SERVER = r"MYDELL23\SQLEXPRESS01"   # your instance
DATABASE = "AdventureWorks"     # change if your DB name differs
DRIVER = "ODBC Driver 17 for SQL Server"  # or "ODBC Driver 18 for SQL Server"


## Creating the SQLAlchemy engine

SQLAlchemy uses an **engine** object as the main entry point for working with SQL databases. Here, an engine is created for SQL Server using the `mssql+pyodbc` dialect with `trusted_connection=yes`, which tells SQL Server to use the current Windows account instead of a username/password.


In [3]:
def create_sqlalchemy_engine_windows(server: str, database: str, driver: str):
    """
    Create a SQLAlchemy engine for SQL Server using Windows Authentication.
    """
    # Connection URL pattern for SQL Server + pyodbc + trusted_connection
    connection_url = (
        f"mssql+pyodbc://@{server}/{database}"
        f"?trusted_connection=yes&driver={driver.replace(' ', '+')}"
    )

    try:
        engine = create_engine(connection_url)
        # Basic connectivity test
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("✅ Connection test successful.")
        return engine
    except Exception as e:
        print("❌ Failed to create engine or connect to the database.")
        print(f"Error details: {e}")
        return None


engine = create_sqlalchemy_engine_windows(SERVER, DATABASE, DRIVER)


✅ Connection test successful.


In [4]:
if engine is None:
    raise RuntimeError("Engine is not initialized. Check the connection settings above.")

query = """
SELECT TOP 10
    SalesOrderID,
    OrderDate,
    CustomerID,
    TotalDue
FROM Sales.SalesOrderHeader
ORDER BY OrderDate DESC;
"""

try:
    df_sample = pd.read_sql(text(query), engine)
    display(df_sample)
    print(f"Retrieved {len(df_sample)} rows.")
except Exception as e:
    print("❌ Error running sample query.")
    print(f"Error details: {e}")


,SalesOrderID,OrderDate,CustomerID,TotalDue
0,75084,2014-06-30,11078,132.6000
1,75085,2014-06-30,11927,18.7187
2,75086,2014-06-30,28789,8.7848
3,75087,2014-06-30,11794,38.6640
4,75088,2014-06-30,14680,125.9258
5,75089,2014-06-30,19585,66.8194
6,75090,2014-06-30,27686,82.8529
7,75091,2014-06-30,20601,88.9194
8,75092,2014-06-30,26564,55.2169
9,75093,2014-06-30,16170,173.0320


Retrieved 10 rows.


### Environment reproducibility

This project uses a Python virtual environment (`venv`) to isolate dependencies. The `requirements.txt` file (created with `pip freeze > requirements.txt`) lists the exact package versions used. To recreate the environment, another user can create a new virtual environment and run `pip install -r requirements.txt`.
